# Radionica 4, dio 2: transfer learning
**AI Lab · GirlTHing**

U prvom dijelu mreža je učila od nule, na 60.000 malih crno-bijelih slika. Sad imamo **prave fotografije odjeće u boji**, ali samo oko 3.000. To je premalo da mreža sama nauči "gledati".

Zato posuđujemo mrežu koju je neko već trenirao na **ImageNetu** (preko milion slika, 1.000 klasa): **MobileNetV2**. Zadržimo sve što zna o ivicama, teksturama i oblicima, a doučimo samo zadnji sloj za naših 10 vrsta odjeće.


### Kako raditi ćelije s TODO
1. Pročitajte tekst **iznad** ćelije: tu piše šta treba uraditi.
2. U ćeliji zamijenite `...` svojim kodom. Ostatak ćelije ne dirajte.
3. Pokrenite ćeliju (Shift + Enter). Ako je sve dobro, nema crvene greške.
4. **Zapele?** Ispod svake TODO ćelije je 💡 **Pomoć**: prvo pročitajte nagovještaj. Rješenje je skriveno ispod njega, otvorite ga tek ako baš ne ide.
5. **Crvena greška?** Pročitajte zadnji red poruke: obično kaže tačno šta fali (npr. zarez ili zagrada).

**Prije početka:** `Runtime → Change runtime type → T4 GPU`

In [ ]:
!git clone -q https://github.com/alexeygrigorev/clothing-dataset-small.git

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

keras.utils.set_random_seed(42)
print('GPU:', tf.config.list_physical_devices('GPU') or 'nema, uključite T4 GPU!')

## 1. Učitamo slike
Svaka klasa je jedan folder (`train/dress`, `train/skirt`, ...). Keras sam pročita foldere i od imena napravi oznake. Sve slike smanjimo na 160×160.

In [ ]:
DIR = 'clothing-dataset-small'
VELICINA = (160, 160)

train_ds = keras.utils.image_dataset_from_directory(
    f'{DIR}/train', image_size=VELICINA, batch_size=32, seed=42)
val_ds = keras.utils.image_dataset_from_directory(
    f'{DIR}/validation', image_size=VELICINA, batch_size=32, shuffle=False)
test_ds = keras.utils.image_dataset_from_directory(
    f'{DIR}/test', image_size=VELICINA, batch_size=32, shuffle=False)

klase_en = train_ds.class_names
prevod = {'dress': 'haljina', 'hat': 'kapa', 'longsleeve': 'dugi rukavi', 'outwear': 'jakna',
          'pants': 'pantalone', 'shirt': 'košulja', 'shoes': 'cipele', 'shorts': 'šorc',
          'skirt': 'suknja', 't-shirt': 'majica'}
klase = [prevod[k] for k in klase_en]
print(klase)

In [ ]:
plt.figure(figsize=(12, 6))
for slike, oznake in train_ds.take(1):
    for i in range(12):
        plt.subplot(2, 6, i + 1)
        plt.imshow(slike[i].numpy().astype('uint8'))
        plt.title(klase[oznake[i]])
        plt.axis('off')
plt.show()

**Pitanje:** po čemu se ove slike razlikuju od Fashion-MNIST-a? Šta mislite, je li ovo lakše ili teže za mrežu?

## 2. Šta gotova mreža već zna?
Prije nego išta treniramo, pitajmo originalni MobileNetV2 šta vidi na jednoj našoj slici. On zna 1.000 ImageNet klasa, među njima i nekoliko vrsta odjeće.

In [ ]:
from tensorflow.keras.applications import mobilenet_v2

imagenet_model = mobilenet_v2.MobileNetV2(weights='imagenet')   # full model, 224x224

putanja = f'{DIR}/test/t-shirt/' + sorted(os.listdir(f'{DIR}/test/t-shirt'))[0]
img = keras.utils.load_img(putanja, target_size=(224, 224))
x = mobilenet_v2.preprocess_input(np.expand_dims(keras.utils.img_to_array(img), 0))

plt.imshow(img); plt.axis('off'); plt.show()
for _, ime, p in mobilenet_v2.decode_predictions(imagenet_model.predict(x, verbose=0), top=5)[0]:
    print(f'{ime:20s} {p:.1%}')

Mreža nikad nije vidjela naš dataset, a ipak zna da je ovo nešto kao majica. Njeni slojevi već prepoznaju tkaninu, kragne, rukave. Mi joj samo trebamo reći **koje su naše klase**.

## 3. Uklonimo stari izlazni sloj i zamrznemo ostatak
Uzimamo MobileNetV2 **bez zadnjeg sloja** (`include_top=False`) i zamrzavamo ga (`trainable = False`): tokom treninga se njegove težine ne mijenjaju.

In [ ]:
baza = mobilenet_v2.MobileNetV2(input_shape=VELICINA + (3,), include_top=False, weights='imagenet')
# Rješenje: zamrznite bazu, da se njene težine ne mijenjaju tokom treninga
baza.trainable = False

print('Slojeva u bazi:', len(baza.layers))
print('Izlaz baze:', baza.output_shape)

> 💡 **Pomoć:** svaki Keras model ima svojstvo `trainable`. Ako ga postavimo na `False`, težine se ne mijenjaju. Pogledajte tekst iznad ćelije.

<details><summary>Pokaži rješenje (samo ako baš zapnete)</summary>

```python
baza.trainable = False
```
</details>

**Izlaz baze je (5, 5, 1280):** 1.280 mapa veličine 5×5. Isto kao u CNN Explaineru, samo mnogo dublje: na kraju je ostala mala slika, ali s jako puno "detektora".

## 4. Novi izlazni sloj za naših 10 klasa
- `Rescaling` pretvori piksele 0–255 u raspon -1 do 1, kako MobileNetV2 očekuje
- `GlobalAveragePooling2D` od svake 5×5 mape uzme prosjek: dobijemo 1.280 brojeva
- `Dense(10, softmax)`: jedan neuron po klasi, kao u prvom dijelu

In [ ]:
ulaz = keras.Input(shape=VELICINA + (3,))
x = layers.Rescaling(1 / 127.5, offset=-1)(ulaz)
x = baza(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
# Rješenje: izlazni sloj: 10 neurona, aktivacija 'softmax'
izlaz = layers.Dense(10, activation='softmax')(x)

model = keras.Model(ulaz, izlaz)
model.summary()

> 💡 **Pomoć:** isti izlazni sloj kao u prvom dijelu: `layers.Dense(10, activation='softmax')`. Ovdje ga "primijenimo" na `x`, isto kao slojeve iznad: `layers.Nesto(...)(x)`.

<details><summary>Pokaži rješenje (samo ako baš zapnete)</summary>

```python
izlaz = layers.Dense(10, activation='softmax')(x)
```
</details>

**Pogledajte na dnu:** *Trainable params* je samo oko 12.800, a ukupno preko 2 miliona. Treniramo manje od 1 % mreže!

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Rješenje: trenirajte 5 epoha na train_ds, s validation_data=val_ds
hist = model.fit(train_ds, validation_data=val_ds, epochs=5)

> 💡 **Pomoć:** isto kao `fit` u prvom dijelu, samo umjesto `X_train, y_train` dajemo cijeli skup `train_ds` (slike i oznake su zajedno), a validaciju kao `validation_data=val_ds`.

<details><summary>Pokaži rješenje (samo ako baš zapnete)</summary>

```python
hist = model.fit(train_ds, validation_data=val_ds, epochs=5)
```
</details>

In [ ]:
plt.plot(hist.history['accuracy'], label='train')
plt.plot(hist.history['val_accuracy'], label='validacija')
plt.xlabel('epoha'); plt.ylabel('accuracy'); plt.legend(); plt.title('Transfer learning')
plt.show()

_, test_acc = model.evaluate(test_ds, verbose=0)
print(f'Accuracy na testu: {test_acc:.1%}')

## 5. Gdje griješi?

In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred = model.predict(test_ds, verbose=0).argmax(axis=1)

fig, ax = plt.subplots(figsize=(9, 9))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=klase).plot(
    ax=ax, cmap='Blues', colorbar=False, xticks_rotation=45)
plt.show()

**Pitanje:** koje klase se miješaju? Je li to isti problem kao u Fashion-MNIST-u?

## 6. Vaša odjeća
Uslikajte nešto što imate na sebi ili u torbi (majicu, cipele, kapu...) i uploadujte. Najbolje radi kad je predmet sam na jednoličnoj podlozi.

In [ ]:
from pathlib import Path
try:
    from google.colab import files
    putanje = list(files.upload())
except ImportError:
    folder = Path('.') if Path('haljina_test1.jpg').exists() else Path('alLab-ml4')
    putanje = [str(folder / ime) for ime in ['haljina_test1.jpg', 'suknja_test1.jpg']]

for fn in putanje:
    img = keras.utils.load_img(fn, target_size=VELICINA)
    p = model.predict(np.expand_dims(keras.utils.img_to_array(img), 0), verbose=0)[0]
    plt.imshow(img); plt.axis('off'); plt.show()
    for i in p.argsort()[::-1][:3]:
        print(f'{klase[i]:12s} {p[i]:.1%}')

**Probajte i zamku:** uploadujte nešto što nije odjeća (šolju, mačku, sebe). Šta model kaže i zašto?

---
## Bonus: šta da smo učile od nule?
Ista količina podataka, ali mala CNN iz prvog dijela, bez ImageNeta. Pokrenite ako imate vremena i uporedite accuracy.

In [ ]:
od_nule = keras.Sequential([
    keras.Input(shape=VELICINA + (3,)),
    layers.Rescaling(1 / 255),
    layers.Conv2D(32, 3, activation='relu'), layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, activation='relu'), layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, activation='relu'), layers.MaxPooling2D(2),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax'),
])
od_nule.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
od_nule.fit(train_ds, validation_data=val_ds, epochs=5)

_, acc = od_nule.evaluate(test_ds, verbose=0)
print(f'Od nule:            {acc:.1%}')
print(f'Transfer learning:  {test_acc:.1%}')

---
**Sjetite se uvodnog primjera s početka:** `vision_learner(dls, resnet18)` i `fine_tune(3)` su radili upravo ovo: uzmi gotovu mrežu, zamijeni joj izlazni sloj, douči na našim slikama. Samo u drugoj biblioteci (fast.ai, na PyTorchu).

### Odgovori na pitanja

Fotografije su u boji, veće su, imaju različite pozadine, osvjetljenje i položaje predmeta; zato je problem složeniji nego Fashion-MNIST. Zamrzavanje baze čuva ImageNet filtere, a trenira se izlaz za deset novih klasa. Miješanje klasa treba očitati iz stvarne matrice konfuzije nakon treninga. Predmet koji nije odjeća ipak dobija neku od deset oznaka jer softmax nema klasu „nepoznato”; velika vjerovatnoća sama ne dokazuje da je ulaz odjeća.